In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA_BWCN (修复版)
#
# 1) 修复“年拼接”：
#    - 从文件名解析 sample_year
#    - 在 preprocess 中重建 time = sample_year 对应的逐日时间
#
# 2) 改为等压面流程：
#    - 先用 hyam/hybm/P0/PS 计算 p_mid
#    - 将 O3(VMR) log-p 插值到等压面（1–100 hPa, 1 hPa 间隔）
#    - 再用等压积分计算 partial O3 column (DU)
#
# 输出：
#   - O3_pc_BWCN_60_90N_{tag}_210yrs.nc
#   - O3_lowest10_years_BWCN_{tag}.txt
#   - O3_lowest25pct_years_BWCN_{tag}.txt
# ============================================================

import os
import re
import glob
import numpy as np
import xarray as xr

# ----------------- 路径 -----------------
ROOT_DIR = "/home/weiji/test_code/plots/New_B2000WCN_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

DATA_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/O3"
FILE_GLOB = os.path.join(DATA_DIR, "B2000WCN.sample.cam.h3.*.O3.hybrid.nc")

# ----------------- 压强范围配置 -----------------
PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

# ----------------- 等压插值目标 -----------------
# 只做 1–100 hPa（1 hPa 间隔），覆盖你两个积分范围
PLEVS_HPA = np.arange(1, 101, 1, dtype=float)

# ----------------- 从路径解析样本年份 -----------------
YEAR_RE = re.compile(r"\.cam\.h3\.([0-9]{4})\.O3\.hybrid\.nc$")

def _parse_sample_year_from_path(path: str) -> int:
    m = YEAR_RE.search(path)
    if not m:
        raise ValueError(f"Cannot parse sample_year from path: {path}")
    return int(m.group(1))

def _get_calendar(ds: xr.Dataset) -> str:
    # WACCM/CAM 常见 noleap/365_day
    cal = None
    try:
        cal = ds["time"].encoding.get("calendar", None)
    except Exception:
        cal = None
    return cal or "noleap"

def preprocess_fix_time_and_sample_year(ds: xr.Dataset) -> xr.Dataset:
    """
    open_mfdataset preprocess:
    - 解析 sample_year
    - 重建 time，使每个年文件的 time 落在自己的 sample_year 上
    - 这样 combine='by_coords' 才不会出现 time 重叠/错乱
    """
    src = ds.encoding.get("source", "")
    sample_year = _parse_sample_year_from_path(src)

    nt = ds.sizes.get("time", None)
    if nt is None:
        return ds

    cal = _get_calendar(ds)

    # 假设每个年文件是逐日数据：
    new_time = xr.cftime_range(
        start=f"{sample_year:04d}-01-01",
        periods=nt,
        freq="D",
        calendar=cal
    )

    ds = ds.assign_coords(time=new_time)
    ds = ds.assign_coords(sample_year=("time", np.full(nt, sample_year, dtype=int)))
    return ds

# ----------------- 计算 p_mid (Pa) -----------------
def compute_pmid(ds: xr.Dataset) -> xr.DataArray:
    """
    p_mid = hyam*P0 + hybm*PS
    返回维度与 O3 对齐的 (time, lev, lat, lon)
    """
    O3 = ds["O3"]
    P0 = ds["P0"]
    PS = ds["PS"]

    if "time" in P0.dims:
        P0 = P0.isel(time=0)
    P0 = P0.squeeze()

    hyam = ds["hyam"]
    hybm = ds["hybm"]

    p_mid = hyam * P0 + hybm * PS
    # broadcast 后可能是 (lev, time, lat, lon)，转成 O3 的维度顺序
    p_mid = p_mid.transpose(*O3.dims)
    return p_mid

# ----------------- log-p 插值到等压面 -----------------
def _logp_interp_1d(p_mid_1d, o3_1d, target_p_pa):
    """
    对单个垂直柱做 log-pressure 1D 插值。
    p_mid_1d, o3_1d: shape (lev,)
    target_p_pa: shape (nplev,)
    """
    p = np.asarray(p_mid_1d)
    o = np.asarray(o3_1d)

    mask = np.isfinite(p) & np.isfinite(o)
    if mask.sum() < 2:
        return np.full(target_p_pa.shape, np.nan, dtype=o.dtype)

    p = p[mask]
    o = o[mask]

    # 按压力升序排序，便于 np.interp
    idx = np.argsort(p)
    p = p[idx]
    o = o[idx]

    # log-p 插值
    lp = np.log(p)
    lt = np.log(target_p_pa)

    return np.interp(lt, lp, o, left=np.nan, right=np.nan)

def interp_o3_to_plev_logp(ds: xr.Dataset, plev_hpa: np.ndarray) -> xr.DataArray:
    """
    将 O3(VMR) 从 hybrid lev 插值到指定等压面 plev(hPa)。
    返回 O3_plev(time, plev, lat, lon)
    """
    O3 = ds["O3"]
    p_mid = compute_pmid(ds)

    target_p_pa = (plev_hpa * 100.0).astype(np.float64)

    O3_plev = xr.apply_ufunc(
        _logp_interp_1d,
        p_mid,
        O3,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p_pa": target_p_pa},
        output_dtypes=[O3.dtype],
    )

    O3_plev = O3_plev.assign_coords(plev=plev_hpa)
    O3_plev.name = "O3"
    return O3_plev

# ----------------- 等压面 partial column (DU) -----------------
def calc_parc_o3_isobaric(O3_plev: xr.DataArray, p_top_hpa: float, p_bot_hpa: float) -> xr.DataArray:
    """
    用等压面 O3(VMR) 计算指定层范围的 partial column (DU)。
    采用 SI 常数 + xarray.integrate(plev)（梯形积分）。
    """
    # ---- constants (SI) ----
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)  # multiply by dp(Pa) -> DU

    # 确保 plev 升序（我们定义的就是 1..100）
    O3_sel = O3_plev.sel(plev=slice(p_top_hpa, p_bot_hpa))

    # integrate 在 hPa 坐标上做 ∫ chi d(plev[hPa])
    integ_hPa = O3_sel.integrate("plev")

    # hPa -> Pa
    integ_Pa = integ_hPa * 100.0

    return integ_Pa * factor  # (time, lat, lon)

# ----------------- BlockA 主体 -----------------
def main_blockA_BWCN():
    files = sorted(glob.glob(FILE_GLOB))
    if not files:
        raise FileNotFoundError(f"No files found with pattern: {FILE_GLOB}")

    print(f"[BlockA_BWCN] Found {len(files)} files.")

    # 用显式文件列表 + preprocess 修复 time
    ds_o3 = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=False,
        preprocess=preprocess_fix_time_and_sample_year,
        chunks={"time": 365, "lat": 96, "lon": 144},
    )

    # 先裁剪到 60–90N 省内存
    ds_polar = ds_o3.sel(lat=slice(60, 90))

    # ===== 关键：先插值到等压面（一次完成，供两个范围复用） =====
    print("[BlockA_BWCN] Interpolating O3 to isobaric levels (1–100 hPa, 1 hPa step) ...")
    O3_plev = interp_o3_to_plev_logp(ds_polar, PLEVS_HPA)

    # ------------- 对每个压强层范围做一套 time series + 极端年 -------------
    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"\n[BlockA_BWCN] ==== Processing range {ptop}-{pbot} hPa ({tag}) ====")

        PC_TS_BWCN_NC = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}_210yrs.nc")
        LOW10_TXT = os.path.join(ROOT_DIR, f"O3_lowest10_years_BWCN_{tag}.txt")
        LOW25_TXT = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_BWCN_{tag}.txt")

        # ------ partial column in DU (time, lat, lon) ------
        print("[BlockA_BWCN] Computing partial column from isobaric O3 ...")
        O3_parc = calc_parc_o3_isobaric(O3_plev, p_top_hpa=ptop, p_bot_hpa=pbot)

        # ------ 极帽平均 (60–90N，lat-lon 加权平均) ------
        weights = np.cos(np.deg2rad(O3_parc.lat))
        var_bwcn = O3_parc.weighted(weights).mean(dim=["lat", "lon"]).load()
        var_bwcn.name = f"O3_pc_BWCN_60_90N_{tag}"

        # 为后续极端年筛选/BlockB 拼接保留 sample_year 坐标
        var_bwcn = var_bwcn.assign_coords(
            sample_year=("time", var_bwcn.time.dt.year.values.astype(int))
        )

        # 保存 time series
        var_bwcn.to_netcdf(PC_TS_BWCN_NC)
        print(f"[BlockA_BWCN] Saved polar-cap partial column TS to: {PC_TS_BWCN_NC}")

        # ------ 基于 spring (3–4 月) 极小值挑选极端年 ------
        print("[BlockA_BWCN] Computing extreme years from Mar–Apr minima ...")

        spring = var_bwcn.sel(time=var_bwcn.time.dt.month.isin([3, 4]))
        spring_min_by_year = spring.groupby("sample_year").min("time")

        spring_years = spring_min_by_year["sample_year"].values.astype(int)
        spring_min_vals = spring_min_by_year.values

        idx_sorted = np.argsort(spring_min_vals)

        n_low10 = min(10, len(spring_years))
        lowest10_years = spring_years[idx_sorted[:n_low10]]

        n_low25 = max(int(0.25 * len(spring_years)), 1)
        lowest25_years = spring_years[idx_sorted[:n_low25]]

        np.savetxt(LOW10_TXT, lowest10_years, fmt="%04d")
        np.savetxt(LOW25_TXT, lowest25_years, fmt="%04d")

        print(f"[BlockA_BWCN] Saved lowest-10 years to: {LOW10_TXT}")
        print(f"[BlockA_BWCN] Saved lowest-25% years to: {LOW25_TXT}")
        print("[BlockA_BWCN] Lowest 10 years (BWCN):", lowest10_years)
        print("[BlockA_BWCN] Lowest 25% years (BWCN):", lowest25_years)

    ds_o3.close()
    print("\n[BlockA_BWCN] Done.")

if __name__ == "__main__":
    main_blockA_BWCN()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (BWCN版 - Nature Style)
#
# 读取 BlockA 输出的 polar-cap partial column TS：
#   - O3_pc_BWCN_60_90N_{tag}_210yrs.nc
# 并基于 lowest10 / lowest25% 画：
#   B1: lowest10 spaghetti vs climatology
#   B2: top3 extremes vs low25% composite
#
# 时段拼接定义：
#   对事件年 Y：
#     Oct–Dec of (Y-1)  +  Jan–Sep of (Y)
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from matplotlib import rcParams

ROOT_DIR = "/home/weiji/test_code/plots/New_B2000WCN_weiji_general_why0008important/General_O3"

TOTAL_DAYS = 365

PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

def get_shifted_lines(da, years_list, days_needed=365):
    """
    用纯 xarray 的时间切片，避免 cftime -> pandas 的不稳定问题。

    对事件年 Y：
      - 取 (Y-1)-10-01 ~ (Y-1)-12-31
      - 取 Y-01-01 ~ Y-09-30
      - 拼接成 365 天
    """
    collected = []
    valid_years = []

    for y in years_list:
        y = int(y)
        prev_y = y - 1

        t_prev_start = f"{prev_y:04d}-10-01"
        t_prev_end   = f"{prev_y:04d}-12-31"
        t_curr_start = f"{y:04d}-01-01"
        t_curr_end   = f"{y:04d}-09-30"

        try:
            seg_prev = da.sel(time=slice(t_prev_start, t_prev_end)).values
            seg_curr = da.sel(time=slice(t_curr_start, t_curr_end)).values
        except Exception:
            continue

        combined = np.concatenate([seg_prev, seg_curr])

        if len(combined) >= days_needed:
            collected.append(combined[:days_needed])
            valid_years.append(y)

    return np.array(collected), np.array(valid_years)

def plot_block_b_nature_style():
    # ================= Nature Style rcParams =================
    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 12,
        "axes.linewidth": 1.2,
        "axes.labelsize": 13,
        "axes.titlesize": 14,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.fontsize": 10,
        "legend.frameon": False,
    })

    # ================= 配色方案 =================
    c_clim_fill = "#999999"
    c_clim_line = "#555555"
    alpha_clim  = 0.30

    c_low25_line = "#004488"
    c_low25_fill = "#6699CC"
    alpha_low25  = 0.40

    spec_colors = ["#D55E00", "#009E73", "#CC79A7"]

    # X 轴设定
    x_axis = np.arange(TOTAL_DAYS)
    tick_pos = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334]
    tick_labels = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                   "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"\n[BlockB] ==== Plotting range {ptop}-{pbot} hPa ({tag}) ====")

        PC_TS_NC  = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}_210yrs.nc")
        LOW10_TXT = os.path.join(ROOT_DIR, f"O3_lowest10_years_BWCN_{tag}.txt")
        LOW25_TXT = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_BWCN_{tag}.txt")

        if not (os.path.exists(PC_TS_NC) and os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT)):
            print("  Missing inputs. Run BlockA first.")
            continue

        var_all = xr.load_dataarray(PC_TS_NC)

        # 事件年列表
        all_years = np.unique(var_all.time.dt.year.values.astype(int))
        lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
        lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

        # --------- Climatology lines (用 all_years[1:] 避免 Y-1 不存在) ---------
        clim_lines, _ = get_shifted_lines(var_all, all_years[1:], days_needed=TOTAL_DAYS)
        clim_mean = np.nanmean(clim_lines, axis=0)
        clim_std  = np.nanstd(clim_lines, axis=0)

        # --------- Lowest 10 lines ---------
        low10_lines, valid_low10_yrs = get_shifted_lines(var_all, lowest10_years, days_needed=TOTAL_DAYS)

        # --------- Lowest 25% composite ---------
        low25_lines, _ = get_shifted_lines(var_all, lowest25_years, days_needed=TOTAL_DAYS)
        low25_mean = np.nanmean(low25_lines, axis=0)
        low25_std  = np.nanstd(low25_lines, axis=0)

        # ============================================================
        # B1. Fig1: Lowest 10 spaghetti vs climatology
        # ============================================================
        fig1, ax1 = plt.subplots(figsize=(12, 7), constrained_layout=True)

        ax1.fill_between(x_axis, clim_mean - clim_std, clim_mean + clim_std,
                         color=c_clim_fill, alpha=alpha_clim,
                         label="Climatology (All Years) ±1$\sigma$", zorder=0)
        ax1.plot(x_axis, clim_mean, color=c_clim_line, lw=2, ls="--", zorder=1)

        ax1.axvline(x=92, color="k", linestyle=":", linewidth=1.0, alpha=0.5)

        colors_spaghetti = cm.plasma(np.linspace(0, 0.85, len(low10_lines)))
        for i, line in enumerate(low10_lines):
            yr = valid_low10_yrs[i]
            ax1.plot(x_axis, line, color=colors_spaghetti[i], alpha=0.8, lw=1.5,
                     label=f"Year {yr}", zorder=2)

        ax1.set_xlim(0, 365)
        ax1.set_xticks(tick_pos)
        ax1.set_xticklabels(tick_labels)
        ax1.set_title(f"Lowest 10 Ozone Events Evolution ({ptop}-{pbot} hPa)\n"
                      f"(Preconditioning: Oct-Dec | Event: Jan-Sep)", pad=15)
        ax1.set_ylabel("Partial Column (DU)")
        ax1.legend(loc="upper right", ncol=2)

        out1 = os.path.join(ROOT_DIR, f"BWCN_O3_lowest10_lines_{tag}_nature.png")
        plt.savefig(out1, dpi=300)
        print("  Saved Fig1:", out1)

        # ============================================================
        # B2. Fig2: Top 3 extremes vs Low 25% composite
        # ============================================================
        fig2, ax2 = plt.subplots(figsize=(8, 5), constrained_layout=True)

        ax2.fill_between(x_axis, clim_mean - clim_std, clim_mean + clim_std,
                         color=c_clim_fill, alpha=alpha_clim,
                         label="Climatology ±1$\sigma$", zorder=0)
        ax2.plot(x_axis, clim_mean, color=c_clim_line, lw=1.5, ls="--", zorder=1)

        ax2.fill_between(x_axis, low25_mean - low25_std, low25_mean + low25_std,
                         color=c_low25_fill, alpha=alpha_low25, linewidth=0,
                         label="Low 25% Composite ±1$\sigma$", zorder=2)
        ax2.plot(x_axis, low25_mean, color=c_low25_line, lw=2.5, zorder=3,
                 label="Low 25% Mean")

        n_top = min(3, len(low10_lines))
        for i in range(n_top):
            yr = valid_low10_yrs[i]
            line = low10_lines[i]
            ax2.plot(x_axis, line, color=spec_colors[i], lw=2.5, zorder=4,
                     label=f"Extr Year {yr}")

        ax2.axvline(x=92, color="k", linestyle=":", linewidth=1.0, alpha=0.5)
        ax2.set_xlim(0, 365)
        ax2.set_xticks(tick_pos)
        ax2.set_xticklabels(tick_labels)
        ax2.set_title(f"Composite Analysis & Extremes ({ptop}-{pbot} hPa)", pad=10)
        ax2.set_ylabel("Partial Column (DU)")
        ax2.legend(loc="best")

        out2 = os.path.join(ROOT_DIR, f"BWCN_O3_composite_vs_extreme_{tag}_nature.png")
        plt.savefig(out2, dpi=300)
        print("  Saved Fig2:", out2)

        plt.close("all")

    print("\n[BlockB] Plotting finished (Nature Style).")

if __name__ == "__main__":
    plot_block_b_nature_style()
